# Create and Upload Drillhole Intervals
---

First import the evo `ServiceManagerWidget` along with necessary typed objects along with other familiar python packages

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

from evo.notebooks import ServiceManagerWidget
from evo.objects.typed import DownholeIntervals, DownholeIntervalsData

pio.templates.default = "simple_white"

%load_ext evo.widgets

## Connect to Seequent Evo
Replace the `<CLIENT ID>` with your own. If you need to create a client id or have forgotten yours, visit the developer portal [my apps](https://developer.seequent.com/my-apps) page.

In [ ]:
evo_connection = await ServiceManagerWidget.with_auth_code(
    client_id="<CLIENT ID>",
).login()

## Create Synthetic Data

For this example, create 3 synthetic drillholes:

In [ ]:
rng = np.random.default_rng(7)
n_holes, n = 3, 20
collars = rng.uniform([0, 0], [200, 200], (n_holes, 2))

pts, ints = [], []
for i, (cx, cy) in enumerate(collars, 1):
    hid = f"DH{i:03d}"
    az, dip = rng.uniform(0, 2 * np.pi), rng.uniform(np.deg2rad(45), np.deg2rad(80))
    md = np.linspace(0, rng.uniform(120, 300), n)
    dx, dy, dz = np.cos(dip) * np.cos(az), np.cos(dip) * np.sin(az), -np.sin(dip)
    xyz = np.c_[cx + md * dx, cy + md * dy, md * dz]
    grade = rng.lognormal(0.1, 0.65, n)

    a, b = xyz[:-1], xyz[1:]
    ints.append(
        pd.DataFrame(
            {
                "hole_id": hid,
                "from": md[:-1],
                "to": md[1:],
                "x_start": a[:, 0],
                "y_start": a[:, 1],
                "z_start": a[:, 2],
                "x_end": b[:, 0],
                "y_end": b[:, 1],
                "z_end": b[:, 2],
                "x_mid": (a[:, 0] + b[:, 0]) / 2,
                "y_mid": (a[:, 1] + b[:, 1]) / 2,
                "z_mid": (a[:, 2] + b[:, 2]) / 2,
                "grade": grade[:-1],
            }
        )
    )

intervals_df = pd.concat(ints, ignore_index=True)
intervals_df.head()

## View the Drillholes
Use plotly for a quick 3d visualization to inspect the drillholes before uploading to Evo.

In [ ]:
fig = px.scatter_3d(
    intervals_df,
    x="x_mid",
    y="y_mid",
    z="z_mid",
    color="grade",
    color_continuous_scale="Viridis",
    hover_name="hole_id",
    title="Synthetic drillholes, colored by grade",
)
for hid, g in intervals_df.groupby("hole_id"):
    fig.add_scatter3d(
        x=g.x_mid,
        y=g.y_mid,
        z=g.z_mid,
        mode="lines",
        line=dict(width=4, color="rgba(80,80,80,0.45)"),
        name=hid,
        showlegend=False,
    )
fig.update_traces(marker=dict(size=4), selector=dict(type="scatter3d", mode="markers"))
fig.update_layout(scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z"), height=700)
fig.show()

## Upload to Seequent Evo

Create the DownholeIntervalsData object from the DataFrame and upload it to Seequent Evo using the `.create` method.

In [ ]:
intervals_df["hole_id"] = pd.Categorical(intervals_df["hole_id"])

dh = await DownholeIntervals.create(
    evo_connection,
    DownholeIntervalsData(
        name="Synthetic angled DH intervals",
        intervals=intervals_df,
        is_composited=True,
        depth_unit="m",
    ),
)

dh